<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Web_Search_API_Tavily.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Web Search APIs: Grounding LLMs with Live Data — and Your First Tool Call

Every model we have used so far answers from **frozen training data** — ask about anything that happened after its cutoff, and it can only guess or refuse. The previous lesson fixed that with Perplexity, an LLM with a search engine already fused in. This lesson takes that machine apart: a **web search API** (we use [Tavily](https://www.tavily.com/)) hands *your code* live results, and *you* decide how they reach the model.

Two patterns, in increasing order of magic:

1. **Search-then-read** — your code searches, places the results into the prompt, and the model answers from them. RAG, with the web as the corpus.
2. **Tool calling** — you *describe* the search function to the model, and **the model decides when to call it**. This is the course's first taste of tool calling — the mechanism that Sections 11 and 12 grow into full agents.

## 🧭 What You'll Learn

- Calling **Tavily** directly with `tavily-python`: cleaned page content, relevance scores, and zero scraping — in one API call
- **Grounding a prompt by hand** with live results, using the course's cited-sources template — web search as just-in-time RAG
- **Native tool calling, side by side on all three providers**: declare a `search_web` tool, read the model's tool call, execute it, return the result — you run the one you picked and read the other two
- A provider-neutral **`answer_with_search()`** helper — a bounded model↔tool loop that is, quietly, a 30-line agent skeleton
- The web-search API landscape as of July 2026: what to use, what retired, and how to choose

## 1. Setup: Environment, Keys, and Providers

The standard course setup cell. **`PROVIDER` drives the whole notebook**: every cell below is gated on it, so exactly one provider's code runs and you only need *that* provider's key — plus one more: `TAVILY_API_KEY`. Section 6 still prints all three dialects for you to read; only the selected one executes.

**Get a Tavily key:** sign up at [app.tavily.com](https://app.tavily.com) — the key appears on the dashboard immediately, no credit card required. The free tier includes 1,000 API credits per month (a basic search costs 1 credit), which covers this notebook many times over (as of July 2026). Add it to Colab Secrets (🔑 icon) or your local `.env`, like the model keys.

Switching provider is a one-line change: set `PROVIDER`, re-run, done — no other edits. (Section 6.3 is the exception: the Claude cell ships commented out, so selecting `"anthropic"` also means un-commenting it.)

The **model field is an editable dropdown** (`{allow-input: true}`): pick one of the listed course defaults, or type any newer model ID straight into the box — no code changes needed. (Locally, simply edit the string.) If that ID does not belong to the provider you selected, the notebook warns and falls back to that provider's course default — so changing `PROVIDER` on its own is always safe.

In [1]:
# ============================================================
# ⚙️ Setup — environment, dependencies, API keys, provider
# ============================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# Pick your model provider (dropdown in Colab; edit the value locally)
PROVIDER = "gemini"  # @param ["gemini", "openai", "anthropic"]

# Pick a model for the selected provider — or TYPE any newer model ID into the
# box (the dropdown is editable thanks to allow-input):
CHAT_MODEL = "gemini-3.7-flash"  # @param ["gemini-3.7-flash","gemini-3.5-flash-lite", "gpt-5.6-luna", "claude-sonnet-5"] {allow-input: true}

# Only the selected provider's key is needed — every cell below is gated on
# PROVIDER, so you never need keys for providers you are not running.
PROVIDER_KEYS = {
    "gemini": "GOOGLE_API_KEY",
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
}
REQUIRED_KEYS = [PROVIDER_KEYS[PROVIDER], "TAVILY_API_KEY"]

if IN_COLAB:
    import importlib
    import site
    import subprocess

    # Shared install profile, pinned course-wide (July 2026). Library updates can
    # change behavior, so we pin versions to keep every cell reproducible.
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "google-genai==2.3.0",
            "openai==2.46.0",
            "anthropic==0.117.0",
            "tavily-python==0.7.26",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without a runtime restart

    # In Colab: Secrets tab (🔑 icon in the left sidebar) → Add new secret →
    # name it e.g. GOOGLE_API_KEY, paste the key, and toggle notebook access on.
    from google.colab import userdata

    for key in REQUIRED_KEYS:
        os.environ[key] = userdata.get(key)

if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

    # Locally: install dependencies once from the repo's requirements file.
    # API keys live in a .env file at the repo root (never hardcode keys in cells).
    from dotenv import load_dotenv

    load_dotenv()
    missing = [k for k in REQUIRED_KEYS if not os.getenv(k)]
    assert not missing, f"Missing from .env: {missing}"

print(f"✅ Setup complete — {'Colab' if IN_COLAB else 'local'} | provider: {PROVIDER}")

✅ Setup complete — local | provider: gemini


## 2. The Course `generate()` Helper

📎 *Unchanged — built in "How To Use LLMs via API". It powers the no-search baseline and the hand-grounded answers below.*

In [2]:
# 📎 generate() and embed() are the course's two provider helpers,
#    built in “How To Use LLMs via API” and “Basic RAG” — unchanged here.
from anthropic import Anthropic
from google import genai
from google.genai import types as genai_types
from openai import OpenAI

# Course-standard default models per provider (August 2026)
MODELS = {
    "gemini": "gemini-3.7-flash",
    "openai": "gpt-5.6-luna",
    "anthropic": "claude-sonnet-5",
}

# The setup-cell form selection (or any typed model ID) overrides the default —
# but only when it belongs to the provider you picked. The two form fields are
# independent, so changing PROVIDER alone would otherwise send, say, a Gemini
# model ID to OpenAI and 400.
MODEL_PREFIXES = {"gemini": "gemini", "openai": "gpt", "anthropic": "claude"}
if CHAT_MODEL.startswith(MODEL_PREFIXES[PROVIDER]):
    MODELS[PROVIDER] = CHAT_MODEL
else:
    print(f"⚠️ CHAT_MODEL {CHAT_MODEL!r} does not look like a {PROVIDER} model — falling back to "
          f"{MODELS[PROVIDER]!r}. Edit CHAT_MODEL if you want a different one.")

# Create only the clients we actually need
if PROVIDER == "gemini":
    gemini_client = genai.Client()
if PROVIDER == "openai":
    openai_client = OpenAI()
if PROVIDER == "anthropic":
    anthropic_client = Anthropic()


def generate(prompt, system=None, model=None):
    """Send one prompt to the selected PROVIDER and return the reply text."""
    if PROVIDER == "gemini":
        response = gemini_client.models.generate_content(
            model=model or MODELS["gemini"],
            contents=prompt,
            config=genai_types.GenerateContentConfig(system_instruction=system),
        )
        return response.text

    if PROVIDER == "openai":
        response = openai_client.responses.create(
            model=model or MODELS["openai"],
            instructions=system,
            input=prompt,
            reasoning={"effort": "none"},
        )
        return response.output_text

    if PROVIDER == "anthropic":
        response = anthropic_client.messages.create(
            model=model or MODELS["anthropic"],
            max_tokens=4096,
            **({"system": system} if system else {}),
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


## 3. The Baseline: What the Model Cannot Know

First, the failure we are fixing. We ask about the current state of a package this course actually uses — the kind of fact that changes monthly and that no training set can pin down. (📎 It is deliberately the same question the Perplexity lesson asked — by the end of this notebook, our own pipeline answers it.)

In [3]:
QUESTION = "What is the most recent stable release of the chromadb Python package, and when did it ship?"

print(generate(
    f"Answer the following question. If you cannot know the answer, say exactly why.\nQuestion: {QUESTION}",
))

I cannot know the absolute most recent stable release of the `chromadb` package or its exact release date because I do not have real-time access to the internet or live package registries like PyPI. 

To find the current latest release and its release date, you can check:
- **PyPI**: [pypi.org/project/chromadb/](https://pypi.org/project/chromadb/)
- **GitHub Releases**: [github.com/chroma-core/chroma/releases](https://github.com/chroma-core/chroma/releases)


**What just happened?** The model answered from training memory: a stale version number, a hedge, or an honest "my information has a cutoff" (**your output may differ** — that variability is itself the symptom). Nothing inside the model updates when PyPI does. Time to hand it a search engine.

## 4. Tavily: Search Built for LLM Pipelines

Classic search APIs return *snippets* — a few teaser words per result — and leave the scraping, cleaning, and ranking to you. Tavily was built for LLM pipelines: one call returns the **extracted page content**, a **relevance score** per result, and optionally a synthesized answer. (Google's Custom Search JSON API, which an earlier version of this lesson used, is closed to new customers.)

In [4]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

response = tavily_client.search(
    query=QUESTION,
    max_results=5,         # Tavily returns up to 20 per call
    search_depth="basic",  # "advanced" digs deeper (2 credits instead of 1, as of July 2026)
)

for r in response["results"]:
    print(f"┌─ score {r['score']:.3f} | {r['title'][:70]}")
    print(f"│  {r['url']}")
    preview = r["content"][:180].replace("\n", " ").strip()
    print(f"│  {preview}…")
    print("└" + "─" * 70)

┌─ score 0.873 | ChromaDB - Wikipedia
│  https://en.wikipedia.org/wiki/ChromaDB
│  | Developer | Chroma | | Release | October 22, 2022; 3 years ago (2022-10-22) | | Stable release | v1.0.15 / July 2, 2025; 13 months ago (2025-07-02).: | |  | | Written in | Rust "…
└──────────────────────────────────────────────────────────────────────
┌─ score 0.640 | Releases · chroma-core/chroma - GitHub
│  https://github.com/chroma-core/chroma/releases
│  Version: `1.5.6`    Git ref: `refs/tags/1.5.6`    Build Date: `2026-04-07T03:01`    PIP Package: `chroma-1.5.6.tar.gz`    Github Container Registry Image: `:1.5.6`    DockerHub Ima…
└──────────────────────────────────────────────────────────────────────
┌─ score 0.500 | Chromadb Vector database - LangChain: Chat with Your Data
│  https://community.deeplearning.ai/t/chromadb-vector-database/575244
│  Going through your query it looked more of environment to module access issue.  I think this is 3 or 4th time I am seeing a learner having issue with v

**What just happened?** One HTTPS call returned ranked, **content-bearing** results — page text already extracted, no HTML cleanup, no second scraping pass. The `score` is Tavily's semantic relevance between your query and the page content: the same "how close is this text to the question" idea as our retrieval scores, just computed on their side.

## 5. Search-Then-Read: Grounding the Prompt by Hand

Web results are just retrieved chunks. So we treat them exactly like chunks: number them, place them into the course's cited-sources template, and let `generate()` answer *from them* — with `[n]` citations that resolve back to URLs.

📎 *`CITED_PROMPT` is the same template built in the Improving-Data-Sources lesson — chunk metadata in, `[n]` references out. Only the corpus changed: it is now the live web.*

In [5]:
CITED_PROMPT = """You are an AI tutor answering strictly from the numbered sources below.

Rules:
- Use ONLY the sources between <SOURCES> and </SOURCES> to answer.
- Cite the sources you use inline as [1], [2], ... after each claim.
- If the sources do not contain the answer, reply that you cannot answer from the provided sources.

<SOURCES>
{sources}
</SOURCES>

Question: {question}
Answer (with [n] citations):"""


def ground_with_search(question, max_results=5):
    """Search the live web, then answer strictly from the results — with [n] citations."""
    results = tavily_client.search(query=question, max_results=max_results)["results"]
    sources_block = "\n\n".join(
        f"[{i}] {r['title']} ({r['url']})\n{r['content']}"
        for i, r in enumerate(results, start=1)
    )
    reply = generate(CITED_PROMPT.format(sources=sources_block, question=question))
    return reply, results


reply, sources = ground_with_search(QUESTION)
print(reply)
print("\nSources cited above resolve to:")
for i, r in enumerate(sources, start=1):
    print(f"  [{i}] {r['title'][:70]}  —  {r['url']}")

Based on the provided sources, the release information is as follows:

* According to the Wikipedia entry for ChromaDB, the listed **stable release** is **v1.0.15**, which was released on **July 2, 2025** [1].
* According to the GitHub release logs, the most recent non-development release package listed is version **1.5.6** (pip package `chroma-1.5.6.tar.gz`), built on **April 7, 2026** (`2026-04-07T03:01`) [2].

Sources cited above resolve to:
  [1] ChromaDB - Wikipedia  —  https://en.wikipedia.org/wiki/ChromaDB
  [2] Releases · chroma-core/chroma - GitHub  —  https://github.com/chroma-core/chroma/releases
  [3] Chromadb Vector database - LangChain: Chat with Your Data  —  https://community.deeplearning.ai/t/chromadb-vector-database/575244
  [4] 1. ChromaDB Tutorial for Beginners - Create, Store, and Query Vectors   —  https://www.youtube.com/watch?v=_Ci1tLMafQs
  [5] Introduction To ChromaDB | Vector Store For Generative AI LLMs  —  https://medium.com/aimonks/introduction-to-chromadb

**What just happened?** Retrieve → augment → generate, except the index belongs to someone else. The answer is current, and every claim carries a citation you can click (**your output will differ** — it reflects the web right now). Note who drove: *your code* decided when to search, what to search for, and what the model got to read. The model just wrote the last paragraph.

That control is a feature — pipelines like the Perplexity lesson's tool-research loop want exactly this determinism. But sometimes you cannot know *in advance* whether a question needs a search. For that, invert control.

## 6. Tool Calling: Let the Model Decide

Tool calling is a four-step dance, and it is the same dance in every provider's API:

1. **Declare** — you describe your function to the model: name, purpose, parameters (as a JSON Schema).
2. **Decide** — the model reads the question and, instead of answering, may reply with a *structured request*: "call `search_web` with `query=...`".
3. **Execute** — *your code* runs the actual function. The model never executes anything itself — it only asks.
4. **Return** — you send the result back; the model writes the final, grounded answer (or asks for another call).

The next three cells write that loop out by hand, one per provider, so you see every message on the wire. **The cell matching your `PROVIDER` runs; the other two skip** — but read all three: the differences are the whole point. Then we wrap it into one helper. First, the tool itself and its schema:

In [6]:
import json


def _plain_query(query: str):
    """Split Google-style `site:` operators out of a query.

    Models often reach for search-engine syntax (`site:pypi.org chromadb`). Tavily
    takes domain filters as a separate argument and rejects a query made *only* of
    operators, so we translate instead of passing it through. Sanitising whatever
    the model hands your tool is part of the job — the model asks, your code decides.
    """
    terms, domains = [], []
    for token in query.split():
        if token.lower().startswith("site:"):
            target = token[5:].strip("/").split("/")
            domains.append(target[0])       # "pypi.org"  → include_domains
            terms.extend(target[1:])        # "project/chromadb" → search terms
        else:
            terms.append(token)
    # An operator-only query leaves nothing to search for: fall back to the domain text.
    return " ".join(terms) or " ".join(domains), domains


def search_web(query: str) -> str:
    """Search the live web and return the top results as compact JSON.

    This plain Python function is our "tool". Each provider gets a machine-readable
    description of it below — and the MODEL decides when it gets called.
    """
    terms, domains = _plain_query(query)
    try:
        results = tavily_client.search(
            query=terms,
            max_results=5,
            include_domains=domains or None,
        )["results"]
    except Exception as e:
        # Hand failures back to the model as data, not as a crashed notebook —
        # it can then rephrase and try again (bounded by MAX_ROUNDS in section 7).
        return json.dumps({"error": f"{type(e).__name__}: {e}"})
    return json.dumps([
        {"title": r["title"], "url": r["url"], "content": r["content"][:800]}
        for r in results
    ])


# One JSON-Schema description, reused by all three providers (only the wrapper differs):
SEARCH_TOOL_SCHEMA = {
    "name": "search_web",
    "description": (
        "Search the live web for current information. Use this whenever the question "
        "involves anything that may have changed after your training data was collected "
        "— versions, releases, prices, leaderboards, news, people in roles."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": (
                    "Plain natural-language search terms, as you would type into a "
                    "search box. Do not use search-engine operators such as site: or "
                    "quotes — this is a semantic search engine, not Google."
                ),
            },
        },
        "required": ["query"],
    },
}

TOOL_QUESTION = "What are the pricing tiers of the Tavily search API, and how many free credits does the free plan include per month?"


### 6.1 Gemini: `function_declarations` → `function_calls` → `Part.from_function_response`

The `google-genai` SDK can actually run this whole loop *for you* (pass the Python function itself as a tool — see the optional experiment at the end). We disable that automation here on purpose: watching the two turns is the lesson.

*Runs when `PROVIDER == "gemini"` — otherwise it skips politely.*

In [7]:
if PROVIDER == "gemini":
    from google import genai
    from google.genai import types as gtypes

    g_client = genai.Client()
    g_config = gtypes.GenerateContentConfig(
        tools=[gtypes.Tool(function_declarations=[SEARCH_TOOL_SCHEMA])],
        # Manual mode: we want to SEE the tool call, not have the SDK hide it
        automatic_function_calling=gtypes.AutomaticFunctionCallingConfig(disable=True),
    )

    # Turn 1 — the model reads the question and (we expect) asks for the tool
    contents = [gtypes.Content(role="user", parts=[gtypes.Part(text=TOOL_QUESTION)])]
    response = g_client.models.generate_content(
        model=MODELS["gemini"], contents=contents, config=g_config)

    if not response.function_calls:
        print("Model answered without the tool:\n", response.text)
    else:
        call = response.function_calls[0]
        print(f"🔧 Model asked for: {call.name}({dict(call.args)})")

        result = search_web(**dict(call.args))  # WE execute — the model only asked

        # Turn 2 — append the model's request + our result, ask again
        contents.append(response.candidates[0].content)
        contents.append(gtypes.Content(role="user", parts=[
            gtypes.Part.from_function_response(name=call.name, response={"result": result}),
        ]))
        response = g_client.models.generate_content(
            model=MODELS["gemini"], contents=contents, config=g_config)

        # Turn 2 is usually the answer — but the model may ask for ANOTHER search
        # instead, in which case there is no text to print. That is precisely why
        # section 7 wraps this exchange in a bounded loop.
        if response.text:
            print("\n" + response.text)
        else:
            follow_up = dict(response.function_calls[0].args) if response.function_calls else {}
            print(f"\n↻ The model wants another search: {follow_up}\n"
                  "  One manual round is not always enough — section 7 loops until it stops.")
else:
    print(f"Skipping — PROVIDER is {PROVIDER!r}, not 'gemini'. "
          "Read the code anyway: the shape is the lesson.")

🔧 Model asked for: search_web({'query': 'Tavily search API pricing tiers free plan credits'})

### Free Plan Credits
The Tavily Search API free tier (**Researcher plan**) includes **1,000 free API credits per month**. No credit card is required to sign up.

---

### Tavily API Pricing Tiers

Tavily uses a credit-based model where:
* **Basic Search:** 1 credit per request
* **Advanced Search:** 2 credits per request
* **Extract:** 1 credit per 5 successful URLs

| Tier / Plan | Monthly Credits | Monthly Price | Effective Price / Credit |
| :--- | :--- | :--- | :--- |
| **Researcher (Free)** | 1,000 | Free ($0) | Free |
| **Project** | 4,000 | $30 / month | $0.0075 |
| **Bootstrap** | 15,000 | $100 / month | $0.0067 |
| **Startup** | 38,000 | $220 / month | $0.0058 |
| **Growth** | 100,000 | $500 / month | $0.0050 |
| **Pay-As-You-Go** | Usage-based | Usage-based | $0.0080 |
| **Enterprise** | Custom | Custom | Custom volume discounts |

*Note: If you exceed your subscription tier's cred

### 6.2 OpenAI: flat tool defs → `function_call` items → `function_call_output`

In the Responses API the tool definition sits flat (`type`, `name`, `description`, `parameters`), tool requests arrive as `function_call` items in `response.output`, and results go back as `function_call_output` items tied together by `call_id`.

*Runs when `PROVIDER == "openai"` — otherwise it skips politely.*

In [8]:
if PROVIDER == "openai":
    from openai import OpenAI

    o_client = OpenAI()
    o_tools = [{"type": "function", **SEARCH_TOOL_SCHEMA}]

    input_items = [{"role": "user", "content": TOOL_QUESTION}]
    response = o_client.responses.create(
        model=MODELS["openai"], input=input_items, tools=o_tools,
        reasoning={"effort": "none"})

    calls = [item for item in response.output if item.type == "function_call"]
    if not calls:
        print("Model answered without the tool:\n", response.output_text)
    else:
        call = calls[0]
        print(f"🔧 Model asked for: {call.name}({call.arguments})")

        result = search_web(**json.loads(call.arguments))  # arguments arrive as a JSON string

        input_items += response.output  # keep the model's request in the history
        input_items.append({
            "type": "function_call_output", "call_id": call.call_id, "output": result,
        })
        response = o_client.responses.create(
            model=MODELS["openai"], input=input_items, tools=o_tools,
            reasoning={"effort": "none"})

        # Turn 2 may be another tool request rather than the answer (see section 7).
        if response.output_text:
            print("\n" + response.output_text)
        else:
            follow_ups = [i for i in response.output if i.type == "function_call"]
            print(f"\n↻ The model wants another search: {follow_ups[0].arguments}\n"
                  "  One manual round is not always enough — section 7 loops until it stops.")
else:
    print(f"Skipping — PROVIDER is {PROVIDER!r}, not 'openai'. "
          "Read the code anyway: the shape is the lesson.")

Skipping — PROVIDER is 'gemini', not 'openai'. Read the code anyway: the shape is the lesson.


### 6.3 Anthropic: `input_schema` → `stop_reason="tool_use"` → `tool_result`

Claude signals a tool request with `stop_reason="tool_use"` and a `tool_use` content block; the result goes back as a `tool_result` block *in a user message*, linked by `tool_use_id`.

*This cell is **commented out** so the notebook runs without an Anthropic key. Have one? Un-comment the cell (Ctrl-/ or Cmd-/) and add `ANTHROPIC_API_KEY` back to `REQUIRED_KEYS` in the setup cell.*

In [9]:
# ⚠️ COMMENTED OUT — this is the only cell that needs an ANTHROPIC_API_KEY.
#    Have one? Set PROVIDER = "anthropic" in the setup cell, then select every
#    line below and toggle the comments off (Ctrl-/ on Windows/Linux, Cmd-/ on
#    macOS). The code is the exact Claude counterpart of the two cells above —
#    read it either way, the shape is the lesson.
print("ℹ️ Anthropic cell is commented out — uncomment it if you have a Claude key.")

# if PROVIDER == "anthropic":
#     from anthropic import Anthropic
#
#     a_client = Anthropic()
#     a_tools = [{
#         "name": SEARCH_TOOL_SCHEMA["name"],
#         "description": SEARCH_TOOL_SCHEMA["description"],
#         "input_schema": SEARCH_TOOL_SCHEMA["parameters"],  # same JSON Schema, different key
#     }]
#
#     messages = [{"role": "user", "content": TOOL_QUESTION}]
#     response = a_client.messages.create(
#         model=MODELS["anthropic"], max_tokens=1024, tools=a_tools, messages=messages)
#
#     if response.stop_reason != "tool_use":
#         print("Model answered without the tool:\n", response.content[0].text)
#     else:
#         call = next(block for block in response.content if block.type == "tool_use")
#         print(f"🔧 Model asked for: {call.name}({call.input})")
#
#         result = search_web(**call.input)  # input is already a dict here
#
#         messages.append({"role": "assistant", "content": response.content})
#         messages.append({"role": "user", "content": [
#             {"type": "tool_result", "tool_use_id": call.id, "content": result},
#         ]})
#         response = a_client.messages.create(
#             model=MODELS["anthropic"], max_tokens=1024, tools=a_tools, messages=messages)
#
#         # Turn 2 may be another tool request rather than the answer (see section 7).
#         answer = next((b.text for b in response.content if b.type == "text"), None)
#         if answer:
#             print("\n" + answer)
#         else:
#             follow_up = next((b.input for b in response.content if b.type == "tool_use"), {})
#             print(f"\n↻ The model wants another search: {follow_up}\n"
#                   "  One manual round is not always enough — section 7 loops until it stops.")
# else:
#     print(f"Skipping — PROVIDER is {PROVIDER!r}, not 'anthropic'. "
#           "Read the code anyway: the shape is the lesson.")

ℹ️ Anthropic cell is commented out — uncomment it if you have a Claude key.


**What just happened?** Three dialects, one grammar:

| Step | Gemini | OpenAI | Anthropic |
|---|---|---|---|
| Declare | `Tool(function_declarations=[…])` | `{"type": "function", …}` in `tools` | `{"name", "description", "input_schema"}` |
| Model asks | `response.function_calls` | `function_call` items in `output` | `tool_use` block, `stop_reason="tool_use"` |
| You return | `Part.from_function_response(…)` | `{"type": "function_call_output", "call_id", "output"}` | `{"type": "tool_result", "tool_use_id", "content"}` |

The differences are spelling. The loop — declare, decide, execute, return — is identical, and in every case **your code holds the keys**: the model requests, you execute.

## 7. `answer_with_search()`: the Provider-Neutral Helper

The same loop, wrapped once for whichever `PROVIDER` you selected — with a hard ceiling on rounds, because an unbounded model-decides-when-to-stop loop is how token bills become incidents. This mirrors `generate()`: three branches, one behavior.

In [10]:
MAX_ROUNDS = 4  # hard ceiling on model↔tool round-trips — never ship an unbounded loop

# What we say when the ceiling is reached. A cap alone only stops the loop; this last
# turn — asked with the tools switched OFF — is what makes it stop with an *answer*.
FINAL_TURN = ("Answer the original question now, using only the search results above. "
              "Do not request another search.")


def answer_with_search(question):
    """Ask the selected PROVIDER, letting it call search_web() whenever it decides to."""
    if PROVIDER == "gemini":
        config = genai_types.GenerateContentConfig(
            tools=[genai_types.Tool(function_declarations=[SEARCH_TOOL_SCHEMA])],
            automatic_function_calling=genai_types.AutomaticFunctionCallingConfig(disable=True),
        )
        contents = [genai_types.Content(role="user", parts=[genai_types.Part(text=question)])]
        for _ in range(MAX_ROUNDS):
            response = gemini_client.models.generate_content(
                model=MODELS["gemini"], contents=contents, config=config)
            if not response.function_calls:
                return response.text
            contents.append(response.candidates[0].content)
            contents.append(genai_types.Content(role="user", parts=[
                genai_types.Part.from_function_response(
                    name=call.name, response={"result": search_web(**dict(call.args))})
                for call in response.function_calls
            ]))
        # Ceiling reached — force an answer out of what we already gathered.
        contents.append(genai_types.Content(
            role="user", parts=[genai_types.Part(text=FINAL_TURN)]))
        return gemini_client.models.generate_content(
            model=MODELS["gemini"], contents=contents,
            config=genai_types.GenerateContentConfig()).text

    if PROVIDER == "openai":
        tools = [{"type": "function", **SEARCH_TOOL_SCHEMA}]
        input_items = [{"role": "user", "content": question}]
        for _ in range(MAX_ROUNDS):
            response = openai_client.responses.create(
                model=MODELS["openai"], input=input_items, tools=tools,
                reasoning={"effort": "none"})
            calls = [item for item in response.output if item.type == "function_call"]
            if not calls:
                return response.output_text
            input_items += response.output
            input_items += [
                {"type": "function_call_output", "call_id": call.call_id,
                 "output": search_web(**json.loads(call.arguments))}
                for call in calls
            ]
        # Ceiling reached — force an answer out of what we already gathered.
        input_items.append({"role": "user", "content": FINAL_TURN})
        return openai_client.responses.create(
            model=MODELS["openai"], input=input_items,
            reasoning={"effort": "none"}).output_text

    if PROVIDER == "anthropic":
        tools = [{"name": SEARCH_TOOL_SCHEMA["name"],
                  "description": SEARCH_TOOL_SCHEMA["description"],
                  "input_schema": SEARCH_TOOL_SCHEMA["parameters"]}]
        messages = [{"role": "user", "content": question}]
        for _ in range(MAX_ROUNDS):
            response = anthropic_client.messages.create(
                model=MODELS["anthropic"], max_tokens=1024, tools=tools, messages=messages)
            if response.stop_reason != "tool_use":
                return next((b.text for b in response.content if b.type == "text"), "")
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": [
                {"type": "tool_result", "tool_use_id": block.id,
                 "content": search_web(**block.input)}
                for block in response.content if block.type == "tool_use"
            ]})
        # Ceiling reached — force an answer out of what we already gathered.
        messages.append({"role": "user", "content": FINAL_TURN})
        response = anthropic_client.messages.create(
            model=MODELS["anthropic"], max_tokens=1024, messages=messages)
        return next((b.text for b in response.content if b.type == "text"), "")

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")

In [11]:
print(answer_with_search(
    "How many parameters does Llama 4 have? List the model variants and their sizes."
))

Meta’s **Llama 4** family adopts a sparse **Mixture-of-Experts (MoE)** architecture, meaning that while the total parameter counts are large, only a fraction of those parameters (the "active" parameters) are engaged per token during inference.

---

### **Llama 4 Model Variants & Sizes**

| Model Variant | Total Parameters | Active Parameters | Experts | Context Length |
| :--- | :--- | :--- | :--- | :--- |
| **Llama 4 Scout** | **109 Billion** (~109B) | **17 Billion** (17B) | 16 Experts | Up to 10M tokens |
| **Llama 4 Maverick** | **400 Billion** (~400B) | **17 Billion** (17B) | 128 Experts | Up to 1M tokens |
| **Llama 4 Behemoth** | **~2 Trillion** (~2T) | **~288 Billion** (288B) | 128 Experts | Frontier / Research scale |

---

### **Key Highlights of the Variants:**

* **Llama 4 Scout (17B active / 109B total):** Highly optimized for efficiency and extreme long-context retrieval, document analysis, and large codebase reasoning with a **10 million token** context window.
* **Llama

**What just happened?** You asked a question; the model noticed it could not know the answer, searched (possibly more than once), and wrote a grounded reply — with your code executing every search and counting every round. Budget it like an engineer: each round is one model call, each search is one Tavily credit, and `MAX_ROUNDS` caps the worst case. Note *how* the loop ends when the cap is reached — one final call with the tools switched off, forcing the model to answer from what it already gathered. A cap alone only stops the loop; that last call is what makes it stop with an **answer**.

📎 *Look at what this helper actually is: a model, a tool list, and a bounded loop. That is the skeleton of every agent in Sections 11 and 12 — they add memory, more tools, and better stopping rules to exactly this code shape.*

## 8. The Web-Search Landscape (as of July 2026)

Choosing a search backend is mostly choosing **who does the reading**:

- **You do the reading** — [Brave Search API](https://brave.com/search/api/): an independent index with cheap raw results; you fetch and clean pages yourself. Maximum control, most plumbing.
- **The API does the reading** — **Tavily** (this lesson): extracted content and relevance scores, ready for a prompt.
- **The model host does the reading** — **Perplexity's Sonar** (previous lesson): search, read, and answer fused into one call. All three chat providers now also ship *built-in* web-search tools you can switch on server-side; the loop you built in this lesson is what those tools do behind the curtain, with the search backend chosen for you.

Two names you will still find in older tutorials, now gone: Microsoft **retired the Bing Search APIs in August 2025**, and **Google's Custom Search JSON API is closed to new customers**. Build against APIs made for machines — not against whatever a consumer search engine exposes this year.

In [12]:
# 🔬 OPTIONAL EXPERIMENT — Gemini's automatic function calling
# google-genai can run the entire declare→decide→execute→return loop for you:
# pass the Python FUNCTION itself as the tool. Compare with the manual loop above.
if PROVIDER == "gemini":
    from google import genai
    from google.genai import types as gtypes

    auto_client = genai.Client()
    auto_response = auto_client.models.generate_content(
        model=MODELS["gemini"],
        contents="Find the newest release of the openai Python SDK and summarize what changed.",
        config=gtypes.GenerateContentConfig(tools=[search_web]),  # the function IS the tool
    )
    print(auto_response.text)
else:
    print("Skipping — this Gemini-only experiment needs PROVIDER == 'gemini'.")

The newest release of the official OpenAI Python SDK is **v3.2.0**.

---

### **Summary of Changes in v3.2.0**

* **Amazon Bedrock Runtime Endpoint Support:** Added support for Bedrock Runtime endpoints (`SDK-290`, PR [#3623](https://github.com/openai/openai-python/pull/3623)).
* **Streaming & Type Updates:** Added shell call streaming events and introduced new service/image type definitions to the API client (PR [#3635](https://github.com/openai/openai-python/pull/3635)).


## 🔑 Key Takeaways

- A web search API gives your pipeline a **live index**; Tavily returns LLM-ready content and relevance scores in one call, so there is no scraping step to build or maintain.
- **Search-then-read is RAG with someone else's index** — the same cited-sources template from the RAG lessons works unchanged on web results, citations included.
- **Tool calling inverts control**: you declare a schema, the model *requests* a call, your code executes it, the model answers from the result. The grammar is identical across Gemini, OpenAI, and Anthropic — only the field names differ.
- The model decides *when*; your code decides *how much*. Bound every loop (`MAX_ROUNDS`), cap every result list, and count credits like tokens.
- As of July 2026: Tavily and Brave are the practical search APIs, Perplexity fuses search with answering, provider-native search tools do the same in-house — and Bing's search APIs are retired, so don't copy 2024 tutorials.
- `answer_with_search()` is a bounded agent loop in miniature — Sections 11 and 12 build on this exact shape.